## Part 1: Setup and Imports

### 📚 What You'll Learn:
- How to set up LangChain with OpenAI
- Understanding the required libraries for email automation
- Initializing the LLM with appropriate parameters

In [1]:
# TODO: Import all required libraries
# HINT: You'll need:
# - os, json, pandas, dotenv, datetime
# - ChatOpenAI from langchain_openai
# - PromptTemplate, ChatPromptTemplate, MessagesPlaceholder from langchain_core.prompts
# - RunnableWithMessageHistory from langchain_core.runnables.history
# - BaseChatMessageHistory, InMemoryChatMessageHistory from langchain_core.chat_history

import os
import json
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime

# LangChain imports (LangChain 1.0.8 + langchain-community for memory)
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory

# Load environment variables
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# Verify API key
if not api_key:
    raise ValueError("❌ ERROR: Please set OPENAI_API_KEY in your .env file")

print("✅ All libraries imported successfully")
print("✅ OpenAI API Key found")
print("\n🎯 Ready to build your AI Email Assistant!\n")

d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All libraries imported successfully
✅ OpenAI API Key found

🎯 Ready to build your AI Email Assistant!



### Initialize the LLM

**Key Parameters:**
- `model`: We use `gpt-4o-mini` for fast, cost-effective email generation
- `temperature`: 0.7 balances creativity with consistency
- `max_tokens`: 500 is sufficient for most professional emails

16k tokens - 20 emails + user question + AI Response = Context Window

In [2]:
# TODO: Initialize ChatOpenAI
# HINT: Use ChatOpenAI() with api_key, model, temperature, and max_tokens parameters

# Step 1: Create the LLM instance
llm = None  # Replace with ChatOpenAI(...)

llm = ChatOpenAI(
    api_key = api_key,
    model = "gpt-4o-mini",
    temperature = 0.7,
    max_tokens = 500
)

# Step 2: Print confirmation
print("✅ OpenAI LLM initialized successfully")
print(f"   Model: gpt-4o-mini")
print(f"   Temperature: 0.7 (balanced creativity)")
print(f"   Max Tokens: 500")

✅ OpenAI LLM initialized successfully
   Model: gpt-4o-mini
   Temperature: 0.7 (balanced creativity)
   Max Tokens: 500


---

## Part 2: Create Sample Email Dataset

### 📚 What You'll Learn:
- How to structure email data for processing (Gmail-compatible format)
- Categorizing emails by type and priority
- Using pandas for data organization

In [3]:
# TODO: Create sample email dataset
# HINT: Create a list of dictionaries, each with:
# - id, sender, sender_name, subject, body, email_type, priority

# Step 1: Create list with 5 email dictionaries
# Create sample email dataset with Indian names and detailed bodies
sample_emails = [
    {
        "id": 1,
        "sender": "rajesh.kumar@techsolutions.in",
        "sender_name": "Rajesh Kumar",
        "subject": "Project Update - Q4 Goals and Team Alignment",
        "body": "Hi team, I hope this email finds you well. As we approach the end of Q3, I wanted to reach out to discuss our Q4 goals and how we can better align our efforts across all departments. We've made significant progress on the cloud migration project, but there are still some challenges with the timeline that need to be addressed. I'd like to schedule a meeting next week to review our current status, identify any bottlenecks, and ensure everyone is on the same page regarding priorities. Please let me know your availability for Tuesday or Wednesday afternoon. Looking forward to a productive discussion.",
        "email_type": "business",
        "priority": "high"
    },
    {
        "id": 2,
        "sender": "priya.sharma@clientcorp.com",
        "sender_name": "Priya Sharma",
        "subject": "Feedback on Proposal - Pricing and Implementation Clarification",
        "body": "Dear team, Thank you so much for sending over the detailed proposal for the enterprise software solution. We've reviewed it thoroughly with our stakeholders, and overall, we're very impressed with the features and timeline you've outlined. However, we do have some questions regarding the pricing model, particularly around the tiered structure and what's included in each tier. Could you please clarify the differences between the Standard and Premium packages? Additionally, we'd like to understand the implementation timeline better - specifically, how the phases would work for our organization size of 500+ employees. We're also interested in knowing if there's any flexibility in customizing certain modules to fit our specific workflow requirements. Would it be possible to schedule a call this week to discuss these points in detail?",
        "email_type": "client",
        "priority": "high"
    },
    {
        "id": 3,
        "sender": "hr@innovatetech.in",
        "sender_name": "Neha Patel - HR Department",
        "subject": "Annual Performance Review Schedule - Action Required",
        "body": "Dear Employee, This is to inform you that your annual performance review has been scheduled for next week, specifically on Thursday, October 12th at 2:00 PM. As part of the preparation process, please complete your self-assessment form and submit it through our HR portal by Monday, October 9th. The self-assessment should include your key accomplishments over the past year, areas where you feel you've grown professionally, challenges you've faced, and your goals for the upcoming year. Additionally, please prepare any documentation that supports your achievements, such as project completion reports, client feedback, or metrics that demonstrate your contributions to the team. Your manager will also be completing a separate evaluation, and both assessments will be discussed during the review meeting. If you have any questions about the process or need assistance accessing the portal, please don't hesitate to reach out to the HR team.",
        "email_type": "hr",
        "priority": "medium"
    },
    {
        "id": 4,
        "sender": "arjun.verma@globalpartners.com",
        "sender_name": "Arjun Verma",
        "subject": "Strategic Partnership Opportunity - AI Solutions Integration",
        "body": "Hello, I hope you're doing well. My name is Arjun Verma, and I'm the Business Development Manager at Global Partners. I've been following your company's impressive work in the AI and machine learning space, particularly your recent launch of the automated customer service platform. We specialize in providing enterprise integration solutions and have a strong presence in the retail and e-commerce sectors across India and Southeast Asia. I believe there could be a fantastic synergy between our companies - specifically, integrating your AI platform with our existing client base could create tremendous value for both organizations. We have over 200 enterprise clients who are actively looking for advanced AI solutions, and your technology seems like a perfect fit. I'd love to explore this opportunity further and discuss how we might structure a mutually beneficial partnership. Would you be available for a brief introductory call sometime next week? I'm flexible with timing and happy to work around your schedule.",
        "email_type": "business",
        "priority": "low"
    },
    {
        "id": 5,
        "sender": "events@indiatechsummit.com",
        "sender_name": "India Tech Summit 2024",
        "subject": "Early Bird Registration Extended - Save 30% on Tech Summit Passes",
        "body": "Greetings Tech Enthusiast! We're excited to announce that due to popular demand, we've extended our Early Bird registration period for the India Tech Summit 2024! You now have until the end of this month to secure your spot at India's largest technology conference and save 30% on all ticket types. The summit will take place from December 15-17, 2024, at the Mumbai Convention Center and will feature over 100 speakers from leading tech companies including Google, Microsoft, Amazon, and top Indian startups. This year's agenda includes keynotes on AI/ML, Cloud Computing, Blockchain, Cybersecurity, and the Future of Work. You'll also have access to hands-on workshops, networking sessions with industry leaders, and an exclusive startup exhibition showcasing the most innovative products in the market. Don't miss this opportunity to learn from the best, expand your professional network, and stay ahead of the technology curve. Register now using code EARLYBIRD30 to claim your discount. Visit our website for the full agenda and speaker lineup. We look forward to seeing you there!",
        "email_type": "notification",
        "priority": "low"
    }
]

# Step 2: Convert to DataFrame for visualization
emails_df = pd.DataFrame(sample_emails)  # TODO: Use pd.DataFrame(sample_emails)

# Step 3: Display the dataset
print("📧 Sample Email Dataset Created:")
print("="* 100)
print(emails_df[["id", 'sender_name', 'subject', 'email_type']].to_string())                 # TODO: Print the DataFrame with selected columns
print("="* 100)
print(f"\n✅ Total emails loaded: {len(emails_df)}")

📧 Sample Email Dataset Created:
   id                 sender_name                                                            subject    email_type
0   1                Rajesh Kumar                       Project Update - Q4 Goals and Team Alignment      business
1   2                Priya Sharma    Feedback on Proposal - Pricing and Implementation Clarification        client
2   3  Neha Patel - HR Department               Annual Performance Review Schedule - Action Required            hr
3   4                 Arjun Verma       Strategic Partnership Opportunity - AI Solutions Integration      business
4   5      India Tech Summit 2024  Early Bird Registration Extended - Save 30% on Tech Summit Passes  notification

✅ Total emails loaded: 5


---

## Part 3: Build Email Reply Chain

### 📚 What You'll Learn:
- How to design effective prompts for email generation
- Using LangChain's LCEL (LangChain Expression Language)
- Creating reusable chains for email processing

1. Context is king: We provode the sender info, email type
2. Clear Instructions
3. Personnalization : user's name, users title, company
4. LCEL Syntax

In [5]:
# TODO: Create email reply prompt template
# HINT: Define a string template with variables in {curly_braces}

# Step 1: Define the prompt template string
email_reply_template = """You are a professional email assistant. Generate a personalized, 
professional email reply based on the following context.

Sender Name: {sender_name}
Email Type: {email_type}
Priority Level: {priority}
Original Email Subject: {subject}
Original Email Body: {body}
Your Name: {your_name}
Your Title: {your_title}
Company: {company}

Generate a professional, personalized reply that:
1. Addresses the sender by name
2. Acknowledge their emails properly
3. Provide a thoughtful response that is relevent to the email type
4. Maintain a professional tone
5. Include the clear call-to-action or next steps

Reply Email: """

# Step 2: Create PromptTemplate object
prompt = PromptTemplate(
    input_variables = ["sender_name", "email_type", "priority","subject", "body", "your_name", "your_title", "company"],
    template = email_reply_template
)

# Step 3: Create chain using LCEL (pipe operator |)
email_chain = prompt | llm

print("✅ Email reply generation chain created successfully")
print("\n🔗 Chain Structure: PromptTemplate | ChatOpenAI")
print("   └─ This is LCEL (LangChain Expression Language) syntax")

✅ Email reply generation chain created successfully

🔗 Chain Structure: PromptTemplate | ChatOpenAI
   └─ This is LCEL (LangChain Expression Language) syntax


### Create the Reply Generation Function

This function wraps our chain in error handling and data formatting.

In [7]:
email_data =     {
        "id": 4,
        "sender": "arjun.verma@globalpartners.com",
        "sender_name": "Arjun Verma",
        "subject": "Strategic Partnership Opportunity - AI Solutions Integration",
        "body": "Hello, I hope you're doing well. My name is Arjun Verma, and I'm the Business Development Manager at Global Partners. I've been following your company's impressive work in the AI and machine learning space, particularly your recent launch of the automated customer service platform. We specialize in providing enterprise integration solutions and have a strong presence in the retail and e-commerce sectors across India and Southeast Asia. I believe there could be a fantastic synergy between our companies - specifically, integrating your AI platform with our existing client base could create tremendous value for both organizations. We have over 200 enterprise clients who are actively looking for advanced AI solutions, and your technology seems like a perfect fit. I'd love to explore this opportunity further and discuss how we might structure a mutually beneficial partnership. Would you be available for a brief introductory call sometime next week? I'm flexible with timing and happy to work around your schedule.",
        "email_type": "business",
        "priority": "low"
    }

In [8]:
email_data.get("sender_name", "")

'Arjun Verma'

In [9]:
email_data['sender_name']

'Arjun Verma'

In [ ]:
Sender Name: {sender_name}
Email Type: {email_type}
Priority Level: {priority}
Original Email Subject: {subject}
Original Email Body: {body}
Your Name: {your_name}
Your Title: {your_title}
Company: {company}

In [ ]:
result.content if hasattr(result, 'content') else str(result)

if "result" has a .content attribute, use that as the reply; otherwise convert that result to a string and 

In [ ]:
# TODO: Create generate_email_reply function
# HINT: Function should invoke the chain and return a structured dict

def generate_email_reply(email_data, your_name="Sarvesh", 
                        your_title="Delivery Manager", company="Learn With Sarvesh"):
    """
    Generate a personalized email reply using LLM
    
    Args:
        email_data (dict): Email information (sender, subject, body, etc.)
        your_name (str): Your name for the signature
        your_title (str): Your job title
        company (str): Your company name
    
    Returns:
        dict: Generated reply with metadata
    """
    try:
        # Step 1: Invoke the chain with email data
        result = email_chain.invoke({
            "sender_name": email_data.get("sender_name", ""),
            "email_type": email_data.get("email_type", ""),
            "priority": email_data.get("priority", ""),
            "subject": email_data.get("subject", ""),
            "body": email_data.get("body", ""),
            "your_name": your_name,
            "your_title": your_title,
            "company": company
        })
        
        # Step 2: Extract content from response
        reply = result.content if hasattr(result, 'content') else str(result)
        
        # Step 3: Return structured response
        # TODO: Add all return fields
        # - original_email_id, from, to, subject, reply_body, generated_at, status
        return {
            'original_email_id': email_data.get("id"),
            "from": your_name,
            "to": email_data.get("sender"),
            "subject": f"Re: {email_data.get('subject')}",
            "reply_body": reply.strip(),
            "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "status": "generated"
        }
    
    except Exception as e:
        # TODO: Handle errors gracefully
        return {"original_email_id": email_data.get("id"), 
                "error": str(e), 
                "status": "failed"}

print("✅ Email reply generation function created successfully")
print("\n📝 Function signature: generate_email_reply(email_data, your_name, your_title, company)")

In [ ]:
email_data =     {
        "id": 4,
        "sender": "arjun.verma@globalpartners.com",
        "sender_name": "Arjun Verma",
        "subject": "Strategic Partnership Opportunity - AI Solutions Integration",
        "body": "Hello, I hope you're doing well. My name is Arjun Verma, and I'm the Business Development Manager at Global Partners. I've been following your company's impressive work in the AI and machine learning space, particularly your recent launch of the automated customer service platform. We specialize in providing enterprise integration solutions and have a strong presence in the retail and e-commerce sectors across India and Southeast Asia. I believe there could be a fantastic synergy between our companies - specifically, integrating your AI platform with our existing client base could create tremendous value for both organizations. We have over 200 enterprise clients who are actively looking for advanced AI solutions, and your technology seems like a perfect fit. I'd love to explore this opportunity further and discuss how we might structure a mutually beneficial partnership. Would you be available for a brief introductory call sometime next week? I'm flexible with timing and happy to work around your schedule.",
        "email_type": "business",
        "priority": "low"
    }

---

## Part 4: Generate Replies for All Emails

### 📚 What You'll Learn:
- Batch processing multiple emails
- Handling successes and failures
- Displaying results in a user-friendly format

In [ ]:
# TODO: Process all emails and generate replies
# HINT: Loop through sample_emails, call generate_email_reply() for each

print("🚀 Generating replies for all emails...\n")
print("=" * 80)

generated_replies = []

# Step 1: Loop through all emails
for idx, email in enumerate(sample_emails, 1):
    # TODO: Print email info
    
    # Step 2: Generate reply
    reply = None  # TODO: Call generate_email_reply(email)
    
    # TODO: Append to generated_replies
    
    # Step 3: Show status and preview
    # TODO: Print success/failure status
    # TODO: Print preview of first 3 lines if successful

# Step 4: Print summary statistics
print("\n" + "=" * 80)
print(f"\n✅ Processing Complete!")
# TODO: Print total, success, and failed counts

### 🔍 View Full Generated Replies

Let's examine the complete replies to see the quality and personalization.

In [ ]:
# TODO: Display detailed view of all generated replies
# HINT: Loop through generated_replies, show original email + generated reply

print("\n📨 DETAILED VIEW OF GENERATED REPLIES\n")
print("=" * 100)

for i, reply in enumerate(generated_replies, 1):
    if reply.get("status") == "generated":
        # TODO: Get original email
        # TODO: Print reply header
        # TODO: Print original email details
        # TODO: Print generated reply details
        pass

print("\n" + "=" * 100)

---

## Part 5: Save Replies to CSV

### 📚 What You'll Learn:
- Data persistence for generated replies
- Using pandas for CSV export
- Creating audit trails for email automation

In [ ]:
# TODO: Create function to save replies to CSV
# HINT: Filter successful replies, convert to DataFrame, save to CSV

def save_replies_to_csv(replies, filename="email_replies_log.csv"):
    """
    Save all generated replies to a CSV file
    
    Args:
        replies (list): List of reply dictionaries
        filename (str): Output CSV filename
    
    Returns:
        DataFrame: Saved replies as a DataFrame
    """
    # TODO: Check if replies list is empty
    
    try:
        # Step 1: Filter successful replies only
        successful_replies = None  # TODO: List comprehension to filter
        
        if successful_replies:
            # Step 2: Convert to DataFrame
            df_replies = None  # TODO: pd.DataFrame(successful_replies)
            
            # Step 3: Save to CSV
            # TODO: df_replies.to_csv(...)
            
            print(f"✅ {len(successful_replies)} replies saved to '{filename}'")
            return df_replies
        else:
            print("❌ No successful replies to save")
            return None
    
    except Exception as e:
        print(f"❌ Error saving replies: {str(e)}")
        return None

# Save the generated replies
replies_df = save_replies_to_csv(generated_replies)

if replies_df is not None:
    print("\n📊 CSV Preview:")
    print("=" * 100)
    # TODO: Print preview of saved DataFrame
    print("=" * 100)

---

## Part 6: Advanced Feature - Tone Personalization

### 📚 What You'll Learn:
- How to customize LLM output with tone parameters
- Creating dynamic prompts with style instructions
- Comparing different communication styles

In [ ]:
# TODO: Define personalization styles and create advanced prompt
# HINT: Dictionary of tone descriptions + new prompt template with {style_description}

# Step 1: Define 5 personalization styles
personalization_styles = {
    "formal": "",  # TODO: Add description
    "friendly": "",  # TODO: Add description
    # TODO: Add brief, detailed, empathetic
}

# Step 2: Create advanced prompt template with tone
advanced_email_template = """You are a professional email assistant. Generate a personalized, 
professional email reply based on the following context.

# TODO: Add all context variables including {style_description} and {tone}

Reply Email:"""

# Step 3: Create function for personalized replies
def generate_personalized_reply(email_data, your_name="Sarah Johnson", 
                               your_title="Project Manager", company="TechCorp", 
                               tone="formal"):
    """
    Generate a personalized email reply with custom tone
    
    Args:
        email_data (dict): Email information
        your_name (str): Your name
        your_title (str): Your title
        company (str): Your company
        tone (str): Desired tone (formal/friendly/brief/detailed/empathetic)
    
    Returns:
        dict: Generated reply with tone information
    """
    # TODO: Get style description from personalization_styles
    # TODO: Create PromptTemplate with tone variables
    # TODO: Create chain and invoke
    # TODO: Return result dict with tone, reply, status
    pass

print("✅ Tone personalization function created")
print("\n📝 Available tones:")
for tone, description in personalization_styles.items():
    print(f"   • {tone.upper()}: {description}")

### 🎯 Demo: Compare Different Tones

Let's generate replies to the same email using different tones to see the impact.

In [ ]:
# TODO: Generate replies with different tones for comparison
# HINT: Pick one email, generate 3 replies with different tones

demo_email = sample_emails[1]  # Client feedback email

print("🎯 TONE COMPARISON DEMO")
print("=" * 100)
# TODO: Print original email details

demo_tones = ["formal", "friendly", "brief"]

for tone in demo_tones:
    # TODO: Generate reply with current tone
    # TODO: Print tone header and reply
    pass

print("\n💡 Notice how the same email gets different responses based on tone!")

---

## Part 7: Conversation Memory - Thread-Aware Replies

### 📚 What You'll Learn:
- Implementing conversation memory in LangChain
- Maintaining context across email threads
- Understanding different memory types

In [ ]:
# TODO: Set up conversation memory with RunnableWithMessageHistory
# HINT: Create storage dict, get_session_history function, memory prompt, chain wrapper

# Step 1: Initialize thread-based message history storage
thread_message_histories = {}  # Dictionary to store message history for each thread

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """
    Retrieve or create a message history for a given session/thread.
    
    Args:
        session_id (str): Unique identifier for the conversation thread
    
    Returns:
        BaseChatMessageHistory: The message history for this session
    """
    # TODO: Check if session exists, create if not
    # TODO: Return InMemoryChatMessageHistory for this session
    pass

# Step 2: Create prompt template with MessagesPlaceholder for history
memory_prompt = None  # TODO: Use ChatPromptTemplate.from_messages([...])

# Step 3: Create base chain
base_chain = None  # TODO: memory_prompt | llm

# Step 4: Wrap chain with message history
chain_with_history = None  # TODO: RunnableWithMessageHistory(...)

# Step 5: Create function to generate reply with memory
def generate_reply_with_memory(
    email_data,
    thread_id=None,
    your_name="Sarah Johnson",
    your_title="Project Manager",
    company="TechCorp",
    tone="formal"
):
    """
    Generate an email reply with conversation memory for thread context.
    """
    # TODO: Determine thread_key
    # TODO: Get previous message count
    # TODO: Invoke chain_with_history with session_id config
    # TODO: Return structured dict with thread_key and memory_turns
    pass

print("✅ Memory-enabled email reply function created")
print("\n📝 Memory Configuration:")
print("   • Type: RunnableWithMessageHistory (modern LangChain pattern)")
print("   • Storage: InMemoryChatMessageHistory per thread")
print("   • Scope: Per-thread (separate memory for each email thread)")

### 🎯 Demo: Multi-Turn Email Thread

Watch how the assistant maintains context across 3 email exchanges in the same thread.

In [ ]:
# TODO: Demonstrate memory with 3-email thread
# HINT: Create 3 related emails, use same thread_id, show how context builds

demo_thread = [
    # TODO: Create 3 related emails with same subject
]

print("🚀 MEMORY DEMO: Same Thread, Multiple Exchanges")
print("=" * 100)
print("\n💡 Watch how each reply references previous exchanges in the thread!\n")

for i, email in enumerate(demo_thread, 1):
    # TODO: Print email details
    # TODO: Generate reply with memory (same thread_id for all)
    # TODO: Print reply with memory_turns count
    pass

# TODO: Show memory statistics
print("\n✅ Demo Complete!")
print("\n💡 Notice how Reply 2 and 3 reference earlier parts of the conversation!")

### 🧠 Understanding Memory Types

| Storage Type | Use Case | Pros | Cons |
|-------------|----------|------|------|
| **InMemoryChatMessageHistory** | Development, short sessions | Fast, simple, no setup | Lost on restart, not scalable |
| **RedisChatMessageHistory** | Production, distributed systems | Persistent, scalable | Requires Redis server |
| **PostgresChatMessageHistory** | Enterprise, data persistence | Durable, queryable | Requires database setup |

### ✅ When to Use Memory

**Use memory when:**
- Handling email threads with multiple back-and-forth exchanges
- Customer support scenarios where context matters
- Follow-up emails that reference previous discussions

**Skip memory when:**
- Processing standalone, unrelated emails
- Batch processing where emails don't form threads
- Memory/cost constraints are critical

---

## 🎓 Summary & Key Takeaways

### ✅ What You Built Today:

1. **Automated Email Reply System**
   - Integrated LangChain with OpenAI for intelligent email generation
   - Designed context-rich prompts with personalization variables
   - Built reusable chains using LCEL (LangChain Expression Language)

2. **Batch Processing Pipeline**
   - Processed multiple emails efficiently with error handling
   - Generated professional, context-aware replies
   - Saved results to CSV for audit trails and analysis

3. **Tone Personalization**
   - Created 5 different communication styles (formal, friendly, brief, detailed, empathetic)
   - Demonstrated dynamic prompt engineering
   - Showed how the same email can be answered differently based on context

4. **Conversation Memory**
   - Implemented thread-aware memory to maintain context
   - Used RunnableWithMessageHistory for context preservation
   - Demonstrated multi-turn conversations with context carryover

---

## 🎉 Congratulations!

You've built a production-ready AI Email Assistant with:
- ✅ LLM-powered reply generation
- ✅ Tone personalization
- ✅ Conversation memory
- ✅ Batch processing
- ✅ Data persistence

**Next week:** We'll build a YouTube Transcript Summarizer using similar LangChain patterns!

---